# MALDI Data Loading

This notebook loads in parallel MALDI data associated with each run in your cohort.

The following data is loaded and saved out:
* **`spectra_df`**: a table listing, for each run, each m/z spectra observed, along with their corresponding intensities (summed across all points in a run). This will be used for peak identification and filtering in the analysis phase.
* **`poslog_df`**: a table listing each spot (pixel) and its corresponding coordinate information. This is needed to map observed spectra and corresponding intensities back to the slide.
* **`thresholds_df`**: an array identifying, for each run, the nth percentile spectral intensity observed at each pixel. This will be needed to compute peak width for coordinate integration across observed signal.
* **`scaling_factor_dict`**: a JSON identifying the TIC scaling factor computing across each run. If TIC normalization is desired, this will be necessary during coordinate integration to normalize across individual spectral intensity readings

In [ ]:
import json
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from maldi_tools import load_maldi_data

Define:
* `maldi_data_paths`: the list of paths to each of the `.d` folders in your cohort
* `tic_normalize`: whether to add TIC normalization or not. **NOTE: TIC normalization happens across individual runs, not the whole cohort. This normalization is computed in using Bruker's SCiLS' formula (which differs slightly from the traditional equation).**

These parameters are then used to compute the data required for the analysis pipeline.

In [ ]:
maldi_data_paths = [
    # list full paths to each .d data folder
    # ex. "/path/to/run/1.d", "/path/to/run/2.d", etc.
]

# whether to turn on TIC normalization, this is highly recommended for most cohorts
tic_normalize = True

spectra_df, poslog_df, thresholds_arr, scaling_factor_dict = load_maldi_data.extract_maldi_run_spectra(maldi_data_paths)

Set `base_dir`, the root folder of your MALDI analysis (done in the `maldi-pipeline.ipynb` notebook).

The spectra, poslog, thresholds, and scaling factors will be saved to the `output/extracted` folder inside your specified `base_dir`.

In [ ]:
# set the desired root folder for MALDI analysis
base_dir = pathlib.Path("/path/to/base/dir")
extraction_dir = base_dir / "output" / "extracted"

spectra_df.to_csv(extraction_dir / "combined_spectra.csv", index=False)
poslog_df.to_csv(extraction_dir / "poslog_info.csv", index=False)
thresholds_ds = thresholds_arr.to_dataset(name="maldi_run")
thresholds_ds.to_netcdf(extraction_dir / "thresholds.xr")
with open(extraction_dir / "tic_scaling_factor.json", "w") as outfile:
    json.dump(scaling_factor_dict, outfile, indent=4)